In [1]:
import os
%pwd

'/home/tuhin/bangla-political-memes-classification/research'

In [2]:
import os
if os.path.basename(os.getcwd()) == 'research':
    os.chdir("../")

In [3]:
%pwd

'/home/tuhin/bangla-political-memes-classification'

In [4]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TextAnalysisConfig:
    root_dir: Path
    input_train_csv: Path
    input_test_csv: Path
    political_words_csv: Path
    output_train_features: Path
    output_test_features: Path

In [5]:
from memeClassifier.constants import *
from memeClassifier.utils.common import read_yaml, create_directories

In [6]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])

    def get_text_analysis_config(self) -> TextAnalysisConfig:
        config = self.config.text_analysis

        create_directories([config.root_dir])

        text_analysis_config = TextAnalysisConfig(
            root_dir=Path(config.root_dir),
            input_train_csv=Path(config.input_train_csv),
            input_test_csv=Path(config.input_test_csv),
            political_words_csv=Path(config.political_words_csv),
            output_train_features=Path(config.output_train_features),
            output_test_features=Path(config.output_test_features)
        )

        return text_analysis_config

In [7]:
import pandas as pd
from memeClassifier import logger

class TextAnalysis:
    def __init__(self, config: TextAnalysisConfig):
        self.config = config

    def load_political_specific_words(self, top_n=500):
        pol_words = pd.read_csv(self.config.political_words_csv)
        if 'ratio' in pol_words.columns:
            pol_words = pol_words.sort_values('ratio', ascending=False)
        all_pol_words = set(pol_words['word'].head(top_n).str.lower().tolist())
        return all_pol_words

    def extract_text_features(self, text, political_specific_words):
        if pd.isna(text) or text == '':
            return {'political_specific_count': 0, 'political_specific_ratio': 0.0}
        
        words = str(text).lower().split()
        word_count = len(words)
        political_specific_matches = sum(1 for word in words if word in political_specific_words)
        political_specific_ratio = political_specific_matches / word_count if word_count > 0 else 0.0
        
        return {
            'political_specific_count': political_specific_matches,
            'political_specific_ratio': political_specific_ratio
        }

    def initiate_text_analysis(self):
        logger.info("Starting text analysis")
        
        train_df = pd.read_csv(self.config.input_train_csv)
        test_df = pd.read_csv(self.config.input_test_csv)
        
        political_specific_words = self.load_political_specific_words()
        logger.info(f"Political-specific words loaded: {len(political_specific_words)}")
        
        logger.info("Extracting features from training data...")
        train_features = train_df['Processed_Text'].apply(
            lambda x: self.extract_text_features(x, political_specific_words)
        )
        train_features_df = pd.DataFrame(train_features.tolist())
        train_with_features = pd.concat([train_df, train_features_df], axis=1)
        train_with_features.to_csv(self.config.output_train_features, index=False)
        logger.info(f"Train features saved to {self.config.output_train_features}")
        
        logger.info("Extracting features from testing data...")
        test_features = test_df['Processed_Text'].apply(
            lambda x: self.extract_text_features(x, political_specific_words)
        )
        test_features_df = pd.DataFrame(test_features.tolist())
        test_with_features = pd.concat([test_df, test_features_df], axis=1)
        test_with_features.to_csv(self.config.output_test_features, index=False)
        logger.info(f"Test features saved to {self.config.output_test_features}")

In [8]:
try:
    config = ConfigurationManager()
    text_analysis_config = config.get_text_analysis_config()
    text_analysis = TextAnalysis(config=text_analysis_config)
    text_analysis.initiate_text_analysis()
except Exception as e:
    raise e

[2026-06-26 19:10:14,637: INFO: common: yaml file: config/config.yaml loaded successfully]
[2026-06-26 19:10:14,639: INFO: common: yaml file: params.yaml loaded successfully]
[2026-06-26 19:10:14,640: INFO: common: Directory created at: artifacts]
[2026-06-26 19:10:14,641: INFO: common: Directory created at: artifacts/text_analysis]
[2026-06-26 19:10:14,641: INFO: 2846411464: Starting text analysis]
[2026-06-26 19:10:14,648: INFO: 2846411464: Political-specific words loaded: 8]
[2026-06-26 19:10:14,648: INFO: 2846411464: Extracting features from training data...]
[2026-06-26 19:10:14,652: INFO: 2846411464: Train features saved to artifacts/text_analysis/train_features.csv]
[2026-06-26 19:10:14,653: INFO: 2846411464: Extracting features from testing data...]
[2026-06-26 19:10:14,656: INFO: 2846411464: Test features saved to artifacts/text_analysis/test_features.csv]
